# ARES-X: rover autónomo con IA híbrida (RNA + lógica difusa + algoritmo genético)

**Entrega individual · Evaluación ev3 · Plazo: mediodía del 10 de noviembre (recorte fuerte).**

**Alcance de esta entrega:**
- Mapa de Marte reproducible por semilla, con obstáculos, barrancos y sitios científicos.
- **RNA** (MLP 32-16) que clasifica libre, obstáculo o barranco con partición por mapas.
- **Lógica difusa** Mamdani con 9 reglas, para la velocidad.
- **Algoritmo genético** con K puntos de paso, para la ruta, sobre el mapa estimado (ev3-d01, ev3-d02).
- **Supervisor** de seguridad con vetos y códigos de motivo, y replanteo con BFS (ev3-d07, opción B).
- **Experimento** de 20 semillas con dos ablaciones (sin RNA, sin difuso).

**Naturaleza:** simulación ficticia con parámetros de referencia. Todo valor sin fuente se marca como *supuesto de simulación*.

**Entorno:** Python 3.11 o superior en CPU, en local (Windows o Linux). No requiere claves ni archivos externos.

**Qué no incluye (recorte):** demás planetas en modo sonda, eventos de sismo y tormenta, interfaz de ipywidgets completa.


In [1]:
# Dependencias. Ejecutar solo si falta alguna (ver requirements.txt).
# !pip install -q scikit-learn matplotlib pandas nbformat


## 1. Módulos: simulador, RNA, difuso, AG y supervisor

Cada celda es un módulo del proyecto, unido en un solo notebook.

In [2]:
# ----- simulador.py -----
"""ARES-X · Fase 1: simulador físico (sin IA).

Todos los valores numéricos son supuestos de simulación, no especificaciones de misión.
Decisión provisional ev3-d02: el mundo real (verdad) queda separado de lo que el rover observa.
"""
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# Tipos de celda del mundo real
LIBRE, OBSTACULO, BARRANCO = 0, 1, 2

# Movimientos cardinales (fila, columna)
# Ocho direcciones (ev3-d29 B). Una diagonal se permite si la celda de destino es transitable.
MOVIMIENTOS = {"N": (-1, 0), "S": (1, 0), "E": (0, 1), "O": (0, -1),
               "NE": (-1, 1), "NO": (-1, -1), "SE": (1, 1), "SO": (1, -1)}



@dataclass(frozen=True)
class PlanetProfile:
    """Perfil de cuerpo celeste. Solo los rodantes admiten rover de ruedas."""

    nombre: str
    modo: str  # "rover" o "sonda"
    descripcion: str

    def admite_rover(self) -> bool:
        return self.modo == "rover"


PERFILES = {
    "Marte": PlanetProfile("Marte", "rover", "Rover principal: barrancos, polvo, hielo, noche (supuesto de simulación)."),
    "Mercurio": PlanetProfile("Mercurio", "rover", "Rover hipotético: radiación, sobrecalentamiento, cráteres (supuesto)."),
    "Venus": PlanetProfile("Venus", "rover", "Rover especial hipotético: temperatura, presión y corrosión (supuesto)."),
    "Tierra": PlanetProfile("Tierra", "rover", "Banco de calibración: lluvia, sismos, deslizamientos (supuesto)."),
    "Júpiter": PlanetProfile("Júpiter", "sonda", "Gigante gaseoso: sonda atmosférica, sin ruedas."),
    "Saturno": PlanetProfile("Saturno", "sonda", "Gigante gaseoso: sonda atmosférica, sin ruedas."),
    "Urano": PlanetProfile("Urano", "sonda", "Gigante helado: sonda atmosférica, sin ruedas."),
    "Neptuno": PlanetProfile("Neptuno", "sonda", "Gigante helado: sonda atmosférica, sin ruedas."),
}


def validar_compatibilidad(planeta: str, modo: str) -> None:
    """RF-01 / CA-07: bloquea un rover de ruedas en un gigante."""
    if planeta not in PERFILES:
        raise ValueError(f"Planeta desconocido: {planeta!r}")
    perfil = PERFILES[planeta]
    if modo == "rover" and not perfil.admite_rover():
        raise ValueError(
            f"{planeta} no tiene superficie sólida para un rover de ruedas. "
            "Use el modo sonda atmosférica."
        )
    if modo not in ("rover", "sonda"):
        raise ValueError(f"Modo desconocido: {modo!r}")


@dataclass
class World:
    """Mundo real: la verdad del simulador. El rover no lo consulta directamente."""

    tipo: np.ndarray            # LIBRE / OBSTACULO / BARRANCO por celda
    elevacion: np.ndarray       # metros simulados
    sitios: dict                # {(fila, col): valor científico}
    refugio: tuple
    inicio: tuple
    semilla: int

    @property
    def forma(self):
        return self.tipo.shape

    def dentro(self, pos) -> bool:
        f, c = pos
        return 0 <= f < self.tipo.shape[0] and 0 <= c < self.tipo.shape[1]

    def transitable(self, pos) -> bool:
        return self.dentro(pos) and self.tipo[pos] == LIBRE

    def marcar_barranco(self, pos) -> None:
        """Evento manual o de simulación: convierte una celda en precipicio."""
        if not self.dentro(pos):
            raise ValueError(f"Celda fuera del mapa: {pos}")
        if pos in (self.inicio, self.refugio):
            raise ValueError("No se puede poner un barranco sobre inicio o refugio.")
        self.tipo[pos] = BARRANCO


def generar_mundo(semilla: int, filas: int = 20, columnas: int = 30,
                  n_obstaculos: int = 40, n_barrancos: int = 6, n_sitios: int = 4) -> World:
    """Genera un mapa reproducible con la semilla dada (RF-02, RF-12)."""
    rng = np.random.default_rng(semilla)
    tipo = np.zeros((filas, columnas), dtype=int)

    inicio = (filas - 1, 0)
    refugio = (0, columnas - 1)

    def celda_libre_aleatoria():
        return (int(rng.integers(0, filas)), int(rng.integers(0, columnas)))

    for _ in range(n_obstaculos):
        p = celda_libre_aleatoria()
        if p not in (inicio, refugio):
            tipo[p] = OBSTACULO
    for _ in range(n_barrancos):
        p = celda_libre_aleatoria()
        if p not in (inicio, refugio):
            tipo[p] = BARRANCO

    # Elevación suavizada: base de altimetría (supuesto de simulación)
    elevacion = np.cumsum(rng.normal(0, 0.2, size=(filas, columnas)), axis=1)

    sitios = {}
    intentos = 0
    while len(sitios) < n_sitios and intentos < 1000:
        intentos += 1
        p = celda_libre_aleatoria()
        if tipo[p] == LIBRE and p not in (inicio, refugio) and p not in sitios:
            sitios[p] = int(rng.integers(1, 10))

    return World(tipo=tipo, elevacion=elevacion, sitios=sitios,
                 refugio=refugio, inicio=inicio, semilla=semilla)


@dataclass
class SensorSuite:
    """Lecturas ruidosas del entorno inmediato. No ve el mundo completo."""

    rng: np.random.Generator
    ruido: float = 0.1          # desviación relativa del ruido gaussiano
    prob_fallo: float = 0.0     # probabilidad de lectura NaN por sensor

    def _ruidoso(self, valor: float, escala: float) -> float:
        if self.rng.random() < self.prob_fallo:
            return float("nan")
        return float(valor + self.rng.normal(0, self.ruido * escala))

    def leer(self, mundo: World, pos, rumbo: str = "N") -> dict:
        """Distancia al obstáculo o barranco más cercano en la dirección de avance."""
        df, dc = MOVIMIENTOS[rumbo]
        distancia = 20  # alcance máximo del sensor de proximidad (m)
        for paso in range(1, 21):
            p = (pos[0] + df * paso, pos[1] + dc * paso)
            if not mundo.dentro(p):
                break
            if mundo.tipo[p] != LIBRE:
                distancia = paso
                break
        altura_actual = mundo.elevacion[pos]
        celdas_barranco = self._barrancos_vecinos(mundo, pos)
        return {
            "proximidad_frontal_m": self._ruidoso(distancia, 1.0),
            "borde_barranco": len(celdas_barranco) > 0,
            "celdas_barranco": celdas_barranco,
            "altimetria_m": self._ruidoso(altura_actual, 0.5),
        }

    @staticmethod
    def _barrancos_vecinos(mundo: World, pos) -> list:
        """LIDAR de borde: celdas de precipicio en los cuatro vecinos."""
        vecinos = []
        for df, dc in MOVIMIENTOS.values():
            p = (pos[0] + df, pos[1] + dc)
            if mundo.dentro(p) and mundo.tipo[p] == BARRANCO:
                vecinos.append(p)
        return vecinos


@dataclass
class Rover:
    """Estado físico del rover."""

    posicion: tuple
    bateria: float = 100.0      # porcentaje
    rumbo: str = "N"
    distancia_recorrida: int = 0
    en_modo_seguro: bool = False
    historial: list = field(default_factory=list)

    def consumir(self, costo: float) -> None:
        # RF-13 / bordes: la batería nunca queda negativa
        self.bateria = max(0.0, self.bateria - costo)


class MissionLogger:
    """Bitácora por paso (RF-08). Exportable a CSV en la fase de entregables."""

    def __init__(self):
        self.filas = []

    def registrar(self, **datos) -> None:
        self.filas.append(datos)

    def a_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.filas)


def paso_fisico(mundo: World, rover: Rover, movimiento: str, costo: float = 1.0) -> bool:
    """Intenta mover el rover una celda. Devuelve True si se movió.

    Reglas: nunca entra en obstáculo ni barranco (CA-02, sección 12); sin batería no se mueve.
    """
    if movimiento not in MOVIMIENTOS:
        raise ValueError(f"Movimiento inválido: {movimiento!r}")
    if rover.en_modo_seguro or rover.bateria <= 0:
        return False
    df, dc = MOVIMIENTOS[movimiento]
    destino = (rover.posicion[0] + df, rover.posicion[1] + dc)
    rover.rumbo = movimiento
    if not mundo.transitable(destino):
        return False
    rover.consumir(costo)
    rover.posicion = destino
    rover.distancia_recorrida += 1
    return True

In [3]:
# ----- rna.py -----
"""ARES-X · Fase 2: red neuronal de percepción (MLP 32-16, ReLU, softmax de 3 clases).

La etiqueta es el tipo REAL de la celda de enfrente (verdad del simulador).
Las características son lecturas ruidosas de los sensores, sin etiquetar con la misma variable.
Partición train/valid/test por MAPAS (semillas distintas), no por filas (sección 10.1).
"""
from dataclasses import dataclass

import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


CLASES = {LIBRE: "libre", OBSTACULO: "obstáculo", BARRANCO: "barranco"}
NOMBRES_RASGOS = ["proximidad_frontal_m", "borde_barranco", "gradiente_altura_m", "vibracion"]

# Media de la vibración según lo que hay delante (supuesto de simulación: débil señal)
VIBRACION_MEDIA = {LIBRE: 0.2, OBSTACULO: 0.6, BARRANCO: 0.4}


def _rasgos_celda(mundo, pos, rumbo, sensor, rng):
    """Vector de rasgos para una celda observada desde pos mirando hacia rumbo."""
    lectura = sensor.leer(mundo, pos, rumbo)
    df, dc = MOVIMIENTOS[rumbo]
    delante = (pos[0] + df, pos[1] + dc)
    if mundo.dentro(delante):
        gradiente = mundo.elevacion[delante] - mundo.elevacion[pos]
        clase = mundo.tipo[delante]
    else:
        gradiente, clase = 0.0, LIBRE
    grad_ruidoso = gradiente + rng.normal(0, 0.05)
    vibracion = VIBRACION_MEDIA[clase] + rng.normal(0, 0.3)
    return [
        lectura["proximidad_frontal_m"],
        float(lectura["borde_barranco"]),
        grad_ruidoso,
        vibracion,
    ], int(clase)


def generar_dataset(semillas, muestras_por_mapa=400, ruido=0.1, prob_fallo=0.0):
    """Muestras etiquetadas a partir de mapas con las semillas indicadas."""
    X, y = [], []
    for semilla in semillas:
        rng = np.random.default_rng(10_000 + semilla)
        mundo = generar_mundo(semilla=semilla)
        sensor = SensorSuite(rng=rng, ruido=ruido, prob_fallo=prob_fallo)
        filas, columnas = mundo.forma
        for _ in range(muestras_por_mapa):
            pos = (int(rng.integers(1, filas - 1)), int(rng.integers(1, columnas - 1)))
            if mundo.tipo[pos] != LIBRE:
                continue
            rumbo = rng.choice(list(MOVIMIENTOS))
            rasgos, clase = _rasgos_celda(mundo, pos, rumbo, sensor, rng)
            X.append(rasgos)
            y.append(clase)
    X = np.nan_to_num(np.array(X, dtype=float), nan=0.0)
    return X, np.array(y)


def particion_por_mapas(total_mapas=60, seed_base=0):
    """Reparte mapas, no muestras: 70 % entrenamiento, 15 % validación, 15 % prueba."""
    semillas = np.arange(seed_base, seed_base + total_mapas)
    n_train = int(0.70 * total_mapas)
    n_valid = int(0.15 * total_mapas)
    return semillas[:n_train], semillas[n_train:n_train + n_valid], semillas[n_train + n_valid:]


@dataclass
class ResultadoRNA:
    predicciones: np.ndarray
    matriz_confusion: np.ndarray
    macro_f1: float
    recall_barranco: float
    precision_base: float   # exactitud de predecir siempre la clase mayoritaria
    error_calibracion: float


class NeuralPerception:
    def __init__(self, semilla=42):
        self.modelo = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                                    max_iter=300, random_state=semilla,
                                    early_stopping=True, validation_fraction=0.15)
        self.escalador = StandardScaler()
        self.entrenado = False

    def entrenar(self, X, y):
        Xs = self.escalador.fit_transform(X)
        self.modelo.fit(Xs, y)
        self.entrenado = True
        return self

    def predecir_proba(self, X):
        if not self.entrenado:
            raise RuntimeError("La RNA no está entrenada.")
        return self.modelo.predict_proba(self.escalador.transform(np.atleast_2d(X)))

    def evaluar(self, X, y) -> ResultadoRNA:
        proba = self.predecir_proba(X)
        pred = self.modelo.classes_[proba.argmax(axis=1)]
        confianza = proba.max(axis=1)
        aciertos = (pred == y).astype(float)
        # Error de calibración esperado (ECE) con 10 contenedores
        bordes = np.linspace(0, 1, 11)
        ece = 0.0
        for lo, hi in zip(bordes[:-1], bordes[1:]):
            m = (confianza > lo) & (confianza <= hi)
            if m.any():
                ece += m.mean() * abs(confianza[m].mean() - aciertos[m].mean())
        clases = list(CLASES)
        mayoritaria = np.bincount(y, minlength=3).argmax()
        return ResultadoRNA(
            predicciones=pred,
            matriz_confusion=confusion_matrix(y, pred, labels=clases),
            macro_f1=float(f1_score(y, pred, labels=clases, average="macro", zero_division=0)),
            recall_barranco=float(((pred == BARRANCO) & (y == BARRANCO)).sum()
                                  / max((y == BARRANCO).sum(), 1)),
            precision_base=float((y == mayoritaria).mean()),
            error_calibracion=float(ece),
        )

    def explicar(self, rasgos):
        """Evidencia visible (sección 10.1): entrada → normalización → probabilidades."""
        x = np.asarray(rasgos, dtype=float)
        normalizada = self.escalador.transform(x.reshape(1, -1))[0]
        proba = self.predecir_proba(x)[0]
        return {
            "entrada": dict(zip(NOMBRES_RASGOS, x.tolist())),
            "normalizada": dict(zip(NOMBRES_RASGOS, np.round(normalizada, 3).tolist())),
            "probabilidades": {CLASES[c]: round(float(p), 3)
                               for c, p in zip(self.modelo.classes_, proba)},
            "confianza": round(float(proba.max()), 3),
        }

In [4]:
# ----- difuso.py -----
"""ARES-X · Fase 3: controlador difuso Mamdani (min–max, centroide).

Entradas: distancia al riesgo (m), batería (%), inclinación (°).
Salida: velocidad en [0, 1]. Todos los rangos y funciones son supuestos de simulación.
"""
from dataclasses import dataclass

import numpy as np


def trapecio(x, a, b, c, d):
    """Membresía trapezoidal. Con b == a o c == d se obtienen formas de hombro."""
    x = np.asarray(x, dtype=float)
    izq = np.where(b > a, (x - a) / max(b - a, 1e-12), np.where(x >= b, 1.0, 0.0))
    der = np.where(d > c, (d - x) / max(d - c, 1e-12), np.where(x <= c, 1.0, 0.0))
    mu = np.minimum(izq, der)
    return np.clip(np.where((x >= a) & (x <= d), mu, 0.0), 0.0, 1.0)


def triangulo(x, a, b, c):
    return trapecio(x, a, b, b, c)


# Universos de discurso (supuestos de simulación)
DISTANCIA_MAX, BATERIA_MAX, INCLINACION_MAX = 20.0, 100.0, 45.0

# Conjuntos lingüísticos por variable (RF-05: tres por variable de entrada)
CONJUNTOS = {
    "distancia": {
        "cerca": lambda x: trapecio(x, 0, 0, 3, 6),
        "media": lambda x: triangulo(x, 4, 10, 16),
        "lejos": lambda x: trapecio(x, 12, 17, 20, 20),
    },
    "bateria": {
        "critica": lambda x: trapecio(x, 0, 0, 15, 30),
        "baja": lambda x: triangulo(x, 20, 40, 60),
        "alta": lambda x: trapecio(x, 50, 70, 100, 100),
    },
    "inclinacion": {
        "plana": lambda x: trapecio(x, 0, 0, 5, 12),
        "media": lambda x: triangulo(x, 8, 18, 28),
        "fuerte": lambda x: trapecio(x, 24, 32, 45, 45),
    },
}

# Conjuntos de salida (velocidad 0..1)
SALIDA = {
    "parar": lambda x: trapecio(x, 0.0, 0.0, 0.05, 0.15),
    "lenta": lambda x: triangulo(x, 0.1, 0.25, 0.4),
    "moderada": lambda x: triangulo(x, 0.35, 0.5, 0.65),
    "rapida": lambda x: trapecio(x, 0.6, 0.8, 1.0, 1.0),
}

# Reglas documentadas (RF-05: al menos ocho). Operador Y = mínimo.
REGLAS = [
    {"id": "R1", "si": [("distancia", "cerca")], "entonces": "parar",
     "motivo": "Obstáculo o barranco muy cerca: detenerse."},
    {"id": "R2", "si": [("bateria", "critica")], "entonces": "lenta",
     "motivo": "Batería crítica: avanzar lento para ahorrar energía."},
    {"id": "R3", "si": [("inclinacion", "fuerte"), ("distancia", "media")], "entonces": "lenta",
     "motivo": "Pendiente fuerte con riesgo a distancia media: cautela."},
    {"id": "R4", "si": [("bateria", "alta"), ("inclinacion", "plana"), ("distancia", "lejos")],
     "entonces": "rapida", "motivo": "Condiciones ideales: velocidad máxima."},
    {"id": "R5", "si": [("bateria", "baja"), ("distancia", "lejos")], "entonces": "moderada",
     "motivo": "Batería baja pero camino libre: velocidad media."},
    {"id": "R6", "si": [("inclinacion", "media"), ("distancia", "lejos")], "entonces": "moderada",
     "motivo": "Pendiente media con camino libre."},
    {"id": "R7", "si": [("bateria", "alta"), ("distancia", "media")], "entonces": "moderada",
     "motivo": "Energía suficiente con riesgo a distancia media."},
    {"id": "R8", "si": [("inclinacion", "fuerte")], "entonces": "lenta",
     "motivo": "Pendiente fuerte: velocidad lenta para evitar vuelco."},
    {"id": "R9", "si": [("distancia", "media"), ("inclinacion", "plana")], "entonces": "moderada",
     "motivo": "Terreno plano con riesgo a distancia media."},
]

# Valor de salida cuando ninguna regla se activa (sección 12: fallback documentado)
SALIDA_FALLBACK = 0.0


@dataclass
class ResultadoDifuso:
    velocidad: float
    membresias: dict          # {variable: {conjunto: grado}}
    reglas_activadas: list    # [(id, fuerza, motivo)]
    agregado: np.ndarray      # conjunto de salida agregado
    universo: np.ndarray      # eje x de la salida
    fallback: bool


class FuzzyController:
    def __init__(self, resolucion: int = 201):
        self.universo = np.linspace(0.0, 1.0, resolucion)

    @staticmethod
    def fusificar(distancia, bateria, inclinacion) -> dict:
        entradas = {"distancia": distancia, "bateria": bateria, "inclinacion": inclinacion}
        return {
            var: {nombre: float(f(np.clip(entradas[var], _rango(var)[0], _rango(var)[1])))
                  for nombre, f in CONJUNTOS[var].items()}
            for var in CONJUNTOS
        }

    def inferir(self, distancia, bateria, inclinacion) -> ResultadoDifuso:
        # RF-13: una entrada NaN no se propaga; se degrada a la salida fallback
        if any(np.isnan(v) for v in (distancia, bateria, inclinacion)):
            mu = {var: {c: 0.0 for c in CONJUNTOS[var]} for var in CONJUNTOS}
            return ResultadoDifuso(velocidad=SALIDA_FALLBACK, membresias=mu, reglas_activadas=[],
                                   agregado=np.zeros_like(self.universo), universo=self.universo,
                                   fallback=True)
        mu = self.fusificar(distancia, bateria, inclinacion)

        activaciones = {}          # conjunto de salida -> lista de fuerzas
        disparadas = []
        for regla in REGLAS:
            fuerza = min(mu[var][conj] for var, conj in regla["si"])
            if fuerza > 0:
                disparadas.append((regla["id"], float(fuerza), regla["motivo"]))
                activaciones.setdefault(regla["entonces"], []).append(fuerza)

        agregado = np.zeros_like(self.universo)
        for conjunto, fuerzas in activaciones.items():
            recortado = np.minimum(max(fuerzas), SALIDA[conjunto](self.universo))
            agregado = np.maximum(agregado, recortado)

        if agregado.sum() == 0:
            velocidad, fallback = SALIDA_FALLBACK, True
        else:
            velocidad = float(np.sum(self.universo * agregado) / np.sum(agregado))
            fallback = False

        return ResultadoDifuso(velocidad=velocidad, membresias=mu, reglas_activadas=disparadas,
                               agregado=agregado, universo=self.universo, fallback=fallback)


def _rango(var: str):
    return {
        "distancia": (0.0, DISTANCIA_MAX),
        "bateria": (0.0, BATERIA_MAX),
        "inclinacion": (0.0, INCLINACION_MAX),
    }[var]

In [5]:
# ----- genetico.py -----
"""ARES-X · Fase 4: algoritmo genético de planificación (ev3-d01: K puntos de paso).

Cromosoma: K celdas (puntos de paso) dentro del mapa. Se decodifica a una ruta contigua
uniendo inicio → p1 → … → pK → refugio con BFS sobre el MAPA ESTIMADO (ev3-d02).
Reparación: si un tramo no tiene camino, el punto se descarta y se cuenta como reparación.

Subproblemas: laberinto (factibilidad del camino por BFS) y viajero (orden de visitas a sitios
científicos, que determina cuándo se obtiene cada valor).
Todos los pesos y costos son supuestos de simulación.
"""
from collections import deque
from dataclasses import dataclass, field

import numpy as np


PESOS_POR_DEFECTO = {"ciencia": 1.0, "distancia": 0.3, "energia": 0.2,
                     "riesgo": 0.8, "tiempo": 0.1}
PENALIZACION_INVIABLE = 1.0     # P(inviabilidad) por violación dura
COSTO_BATERIA_PASO = 1.0        # % de batería por celda (supuesto de simulación)


def mapa_estimado(mundo: World, rng: np.random.Generator, p_detectar: float = 0.8,
                  p_prior: float = 0.05) -> np.ndarray:
    """Riesgo estimado por celda en [0, 1] que el rover construye con sus sensores.

    Un peligro detectado vale 1.0; uno no detectado queda con el prior bajo p_prior.
    Las celdas libres también tienen prior, porque el rover no conoce el mapa completo.
    """
    riesgo = np.full(mundo.forma, p_prior, dtype=float)
    peligros = np.argwhere(mundo.tipo != LIBRE)
    for f, c in peligros:
        if rng.random() < p_detectar:
            riesgo[f, c] = 1.0
    return riesgo


def camino_bfs(origen, destino, transitable):
    """Camino más corto en 8 direcciones, o None si no existe (laberinto)."""
    if origen == destino:
        return [origen]
    filas, columnas = transitable.shape
    previo = {origen: None}
    cola = deque([origen])
    while cola:
        actual = cola.popleft()
        for df, dc in MOVIMIENTOS.values():
            vecino = (actual[0] + df, actual[1] + dc)
            if not (0 <= vecino[0] < filas and 0 <= vecino[1] < columnas):
                continue
            if vecino in previo or not transitable[vecino]:
                continue
            previo[vecino] = actual
            if vecino == destino:
                camino = [destino]
                while previo[camino[-1]] is not None:
                    camino.append(previo[camino[-1]])
                return camino[::-1]
            cola.append(vecino)
    return None


@dataclass
class Decodificacion:
    ruta: list
    reparaciones: int
    inviable_sin_camino: bool
    orden_sitios: list          # sitios científicos en el orden en que se visitan


@dataclass
class Desglose:
    fitness: float
    ciencia: float
    distancia: float
    energia: float
    riesgo: float
    tiempo: float
    penalizacion: float
    reparaciones: int
    ruta_pasos: int


@dataclass
class ResultadoAG:
    mejor: list
    mejor_desglose: Desglose
    mejor_ruta: list
    historial: list             # por generación: mejor y media de fitness
    candidatos: list = field(default_factory=list)   # top 3 distintos (cromosoma, desglose)


class GeneticPlanner:
    def __init__(self, mundo: World, riesgo: np.ndarray, bateria: float = 100.0,
                 k_puntos: int = 4, poblacion: int = 30, generaciones: int = 30,
                 prob_cruce: float = 0.8, prob_mutacion: float = 0.2, elite: int = 2,
                 torneo: int = 3, pesos=None, semilla: int = 42):
        if poblacion < 30 or generaciones < 30:
            raise ValueError("La consigna pide población ≥ 30 y generaciones ≥ 30.")
        self.mundo = mundo
        self.riesgo = riesgo
        self.bateria = bateria
        self.k = k_puntos
        self.poblacion = poblacion
        self.generaciones = generaciones
        self.prob_cruce = prob_cruce
        self.prob_mutacion = prob_mutacion
        self.elite = elite
        self.torneo = torneo
        self.pesos = {**PESOS_POR_DEFECTO, **(pesos or {})}
        self.rng = np.random.default_rng(semilla)
        self.transitable = riesgo < 0.5   # el rover solo cruza lo que no estima peligroso
        self.transitable[mundo.inicio] = True
        self.transitable[mundo.refugio] = True
        self.pasos_maximos = mundo.forma[0] * mundo.forma[1]
        self.valor_total = sum(mundo.sitios.values()) or 1

    # ---------- decodificación ----------
    def decodificar(self, cromosoma) -> Decodificacion:
        ruta = [self.mundo.inicio]
        reparaciones = 0
        visitados = []
        actual = self.mundo.inicio
        for punto in list(cromosoma) + [self.mundo.refugio]:
            if punto == actual:
                continue
            if punto == self.mundo.refugio:
                tramo = camino_bfs(actual, punto, self.transitable)
                if tramo is None:
                    return Decodificacion(ruta, reparaciones, True, visitados)
            else:
                tramo = camino_bfs(actual, punto, self.transitable)
                if tramo is None:        # reparación: descartar el punto
                    reparaciones += 1
                    continue
            ruta.extend(tramo[1:])
            actual = punto
            if punto in self.mundo.sitios and punto not in visitados:
                visitados.append(punto)
        return Decodificacion(ruta, reparaciones, False, visitados)

    # ---------- evaluación ----------
    def evaluar(self, cromosoma) -> Desglose:
        d = self.decodificar(cromosoma)
        pasos = len(d.ruta) - 1
        penal = d.reparaciones * 0.5 * PENALIZACION_INVIABLE
        if d.inviable_sin_camino:
            penal += PENALIZACION_INVIABLE
        if pasos * COSTO_BATERIA_PASO > self.bateria:
            penal += PENALIZACION_INVIABLE    # batería insuficiente para llegar a zona segura
        ciencia = sum(self.mundo.sitios[s] for s in d.orden_sitios) / self.valor_total
        dist = pasos / self.pasos_maximos
        energia = min(1.0, pasos * COSTO_BATERIA_PASO / self.bateria)
        riesgo = float(np.mean([self.riesgo[c] for c in d.ruta])) if d.ruta else 1.0
        giros = sum(1 for a, b, c in zip(d.ruta, d.ruta[1:], d.ruta[2:])
                    if (b[0] - a[0], b[1] - a[1]) != (c[0] - b[0], c[1] - b[1]))
        tiempo = giros / max(pasos, 1)
        w = self.pesos
        fitness = (w["ciencia"] * ciencia - w["distancia"] * dist - w["energia"] * energia
                   - w["riesgo"] * riesgo - w["tiempo"] * tiempo - penal)
        return Desglose(fitness, ciencia, dist, energia, riesgo, tiempo, penal,
                        d.reparaciones, pasos)

    # ---------- operadores ----------
    def _individuo_aleatorio(self):
        libres = np.argwhere(self.transitable)
        candidatos = list(self.mundo.sitios) + [tuple(p) for p in libres]
        idx = self.rng.choice(len(candidatos), size=self.k, replace=False)
        return [candidatos[i] for i in idx]

    def _seleccion(self, poblacion, fitness):
        indices = self.rng.choice(len(poblacion), size=self.torneo, replace=False)
        ganador = max(indices, key=lambda i: fitness[i])
        return list(poblacion[ganador])

    def _cruce(self, a, b):
        """Cruce de un punto: prefijo de a, sufijo de b."""
        if self.rng.random() > self.prob_cruce or self.k < 2:
            return list(a), list(b)
        punto = int(self.rng.integers(1, self.k))
        return a[:punto] + b[punto:], b[:punto] + a[punto:]

    def _mutar(self, individuo):
        ind = list(individuo)
        for i in range(self.k):
            if self.rng.random() < self.prob_mutacion:
                libres = np.argwhere(self.transitable)
                ind[i] = tuple(libres[self.rng.integers(len(libres))])
        return ind

    # ---------- bucle evolutivo ----------
    def optimizar(self) -> ResultadoAG:
        poblacion = [self._individuo_aleatorio() for _ in range(self.poblacion)]
        historial = []
        desgloses = {}

        for gen in range(self.generaciones):
            fitness = []
            for ind in poblacion:
                clave = tuple(ind)
                if clave not in desgloses:
                    desgloses[clave] = self.evaluar(ind)
                fitness.append(desgloses[clave].fitness)
            fitness = np.array(fitness)
            historial.append({"generacion": gen, "mejor": float(fitness.max()),
                              "media": float(fitness.mean())})

            orden = np.argsort(-fitness)
            nueva = [list(poblacion[i]) for i in orden[:self.elite]]   # elitismo
            while len(nueva) < self.poblacion:
                p1 = self._seleccion(poblacion, fitness)
                p2 = self._seleccion(poblacion, fitness)
                h1, h2 = self._cruce(p1, p2)
                nueva.append(self._mutar(h1))
                if len(nueva) < self.poblacion:
                    nueva.append(self._mutar(h2))
            poblacion = nueva

        for ind in poblacion:
            clave = tuple(ind)
            if clave not in desgloses:
                desgloses[clave] = self.evaluar(ind)
        ordenados = sorted(desgloses.items(), key=lambda kv: -kv[1].fitness)
        mejor_clave, mejor_des = ordenados[0]
        mejor_ruta = self.decodificar(list(mejor_clave)).ruta
        candidatos = [(list(k), v) for k, v in ordenados[:3]]
        return ResultadoAG(mejor=list(mejor_clave), mejor_desglose=mejor_des,
                           mejor_ruta=mejor_ruta, historial=historial, candidatos=candidatos)

In [6]:
# ----- integracion.py -----
"""ARES-X · Fase 5: integración híbrida, supervisor de seguridad y máquina de estados.

Pseudociclo por paso (sección 11):
observar → predecir riesgo (RNA) → inferir velocidad (difuso) → supervisar →
replanificar (AG) si corresponde → actuar → registrar.

El supervisor evalúa primero las prohibiciones físicas y después combina las recomendaciones.
Cada veto lleva un código de motivo reproducible.
"""
from dataclasses import dataclass, field, replace
from functools import lru_cache

import numpy as np


ESTADOS_TERMINALES = {"EXITO", "FRACASO", "MODO_SEGURO", "SIN_RUTA"}
UMBRAL_RIESGO_RNA = 0.5           # probabilidad a partir de la cual la RNA veta
UMBRAL_VELOCIDAD_MINIMA = 0.2     # por debajo de esta velocidad el rover se detiene
GRADOS_POR_GRADIENTE = 20.0       # conversión supuesta de gradiente de altura a grados
DISTANCIA_BLOQUEO_RUTA = 6       # celdas de ruta adelante que se revisan por un obstáculo visto
MARGEN_ENERGIA = 15              # % de batería de reserva sobre el camino al refugio
PASOS_PARADO_MAXIMO = 3          # pasos detenido seguidos antes de replanificar


@dataclass
class Decision:
    accion: str                   # "avanzar", "detener", "replanificar" o "modo_seguro"
    velocidad: float
    motivos: list = field(default_factory=list)


class SafetySupervisor:
    """Veto con prioridad: primero la seguridad física, luego las recomendaciones."""

    def decidir(self, prob_clases: dict, velocidad: float, lectura: dict, bateria: float) -> Decision:
        if np.isnan(lectura["proximidad_frontal_m"]) or np.isnan(lectura["altimetria_m"]):
            return Decision("modo_seguro", 0.0, ["V-SENSOR-NAN"])
        if bateria <= 0:
            return Decision("modo_seguro", 0.0, ["V-BATERIA-0"])
        if lectura["borde_barranco"]:
            return Decision("replanificar", 0.0, ["V-BORDE-BARRANCO"])
        if prob_clases.get("barranco", 0.0) > UMBRAL_RIESGO_RNA:
            return Decision("replanificar", 0.0, ["V-RNA-BARRANCO"])
        if prob_clases.get("obstáculo", 0.0) > UMBRAL_RIESGO_RNA:
            return Decision("replanificar", 0.0, ["V-RNA-OBSTACULO"])
        if velocidad < UMBRAL_VELOCIDAD_MINIMA:
            return Decision("detener", velocidad, ["D-DIFUSO-LENTO"])
        return Decision("avanzar", velocidad, ["OK"])


@lru_cache(maxsize=1)
def rna_entrenada(semilla: int = 42) -> NeuralPerception:
    """RNA entrenada una sola vez por proceso (partición por mapas, tamaño reducido para rapidez)."""
    tr, _, _ = particion_por_mapas(total_mapas=30)
    X, y = generar_dataset(tr, muestras_por_mapa=300)
    return NeuralPerception(semilla=semilla).entrenar(X, y)


class Mision:
    def __init__(self, mundo: World, rna: NeuralPerception = None, semilla: int = 0,
                 bateria: float = 100.0, k_puntos: int = 3, ruido: float = 0.1,
                 eventos: dict = None, usar_rna: bool = True, usar_difuso: bool = True,
                 intervencion_rna=None, intervencion_difuso: float = None):
        self.mundo = mundo
        self.usar_rna = usar_rna
        self.usar_difuso = usar_difuso
        # Intervención causal (ARX-10, ev3-d32 A). Apagada por defecto: None no cambia nada.
        self.intervencion_rna = intervencion_rna          # función(probs) -> probs
        self.intervencion_difuso = intervencion_difuso    # velocidad fija en [0, 1]
        self.rna = rna or rna_entrenada()
        self.rng = np.random.default_rng(semilla)
        self.semilla = semilla
        self.sensor = SensorSuite(rng=self.rng, ruido=ruido)
        self.rover = Rover(posicion=mundo.inicio, bateria=bateria)
        self.riesgo = mapa_estimado(mundo, self.rng)
        self.difuso = FuzzyController()
        self.supervisor = SafetySupervisor()
        self.k = k_puntos
        self.eventos = eventos or {}
        self.log = MissionLogger()
        self.estado = "EXPLORACION"
        self.paso = 0
        self.ruta = []
        self.visitados = set()
        self.replanes = 0
        self.vetos = 0
        self.colisiones = 0
        self.paros_seguidos = 0
        self.objetivos = []           # orden de visita fijado por el AG en la planificación inicial
        self.bateria_inicial = bateria
        self._planear()

    # ---------- planificación ----------
    def _planear(self):
        """Planificación inicial con AG (una sola vez). Después solo BFS (ev3-d07, opción B)."""
        self.replanes += 1
        origen = self.rover.posicion
        if not self.objetivos:
            pendientes = {p: v for p, v in self.mundo.sitios.items() if p not in self.visitados}
            ruta = self._optimizar_ag(origen, pendientes)
            self.objetivos = [c for c in ruta if c in self.mundo.sitios] + [self.mundo.refugio]
            if pendientes and (not ruta or len(ruta) - 1 + MARGEN_ENERGIA > self.rover.bateria):
                self.objetivos = [self.mundo.refugio]
                ruta = []
        else:
            ruta = []
        if not ruta:
            ruta = self._ruta_bfs(origen)
        self.ruta = ruta if camino_valido(ruta, self.riesgo) else []
        return bool(self.ruta) and self.ruta[-1] == self.mundo.refugio

    def _ruta_bfs(self, origen):
        """Une caminos BFS desde el punto actual por los objetivos pendientes, en el orden del AG."""
        pendientes = [o for o in self.objetivos if o not in self.visitados]
        if pendientes and pendientes[-1] != self.mundo.refugio:
            pendientes.append(self.mundo.refugio)
        if not pendientes:
            pendientes = [self.mundo.refugio]
        transitable = self.riesgo < 0.5
        ruta, actual = [origen], origen
        for destino in pendientes:
            tramo = camino_bfs(actual, destino, transitable)
            if tramo is None:
                return []
            ruta.extend(tramo[1:])
            actual = destino
        # Energía: si el recorrido no cabe con margen, se descartan los sitios y se va al refugio
        if len(ruta) - 1 + MARGEN_ENERGIA > self.rover.bateria and self.mundo.refugio != origen:
            tramo = camino_bfs(origen, self.mundo.refugio, transitable)
            return tramo or []
        return ruta

    def _optimizar_ag(self, origen, pendientes):
        if not pendientes:
            return camino_bfs(origen, self.mundo.refugio, self.riesgo < 0.5) or []
        desde = replace(self.mundo, inicio=origen, sitios=pendientes)
        gp = GeneticPlanner(desde, self.riesgo, bateria=max(self.rover.bateria, 1.0),
                            k_puntos=self.k, poblacion=30, generaciones=30,
                            semilla=self.semilla)
        return gp.optimizar().mejor_ruta

    # ---------- pseudociclo ----------
    def paso_mision(self):
        if self.estado in ESTADOS_TERMINALES:
            return self.estado
        self.paso += 1
        if self.paso in self.eventos:
            self.eventos[self.paso](self)

        pos = self.rover.posicion
        if pos == self.mundo.refugio and not self._quedan_sitios():
            self.estado = "EXITO"
            return self.estado
        if len(self.ruta) < 2:
            if not self._planear():
                self.estado = "SIN_RUTA"
                return self.estado
            self._registrar("replanificar", 0.0, ["REPLAN-RUTA-VACIA"], {}, None)
            return self.estado

        siguiente = self.ruta[1]
        rumbo = _rumbo(pos, siguiente)
        # observar: sensores y rasgos para la RNA
        rasgos, _ = _rasgos_celda(self.mundo, pos, rumbo, self.sensor, self.rng)
        lectura = self.sensor.leer(self.mundo, pos, rumbo)
        probs = self.rna.explicar(rasgos)["probabilidades"] if self.usar_rna else {"libre": 1.0}
        if self.intervencion_rna is not None and self.usar_rna:
            probs = self.intervencion_rna(probs)
        self._actualizar_mapa_estimado(pos, rumbo, lectura["proximidad_frontal_m"])
        for celda in lectura["celdas_barranco"]:
            self.riesgo[celda] = 1.0
        # el borde de barranco solo veta si el precipicio está en la siguiente celda de la ruta
        lectura = dict(lectura, borde_barranco=siguiente in lectura["celdas_barranco"])
        # inferir velocidad
        inclinacion = min(45.0, abs(rasgos[2]) * GRADOS_POR_GRADIENTE)
        # la velocidad depende de la distancia a lo largo de la ruta, no solo de lo que hay al frente
        fz = self.difuso.inferir(self._distancia_a_riesgo_en_ruta(), self.rover.bateria, inclinacion)
        if self.intervencion_difuso is not None and self.usar_difuso:
            fz = ResultadoDifuso(velocidad=float(self.intervencion_difuso), membresias={},
                                 reglas_activadas=[], agregado=None, universo=None, fallback=True)
        if not self.usar_difuso:          # ablación: velocidad fija, sin controlador difuso
            fz = ResultadoDifuso(velocidad=0.5, membresias={}, reglas_activadas=[],
                                 agregado=None, universo=None, fallback=True)
        # supervisar
        dec = self.supervisor.decidir(probs, fz.velocidad, lectura, self.rover.bateria)
        # si la siguiente celda de la ruta ya es peligrosa, el plan está roto: replanificar
        if self.riesgo[siguiente] >= 0.5:
            dec = Decision("replanificar", 0.0, ["V-RUTA-CON-RIESGO"])
        # un obstáculo cercano en la ruta invalida el plan: se replanifica, no se espera
        distancia = lectura["proximidad_frontal_m"]
        if dec.accion == "detener" and self._obstaculo_en_ruta(pos, rumbo, distancia):
            dec = Decision("replanificar", 0.0, ["V-OBSTACULO-EN-RUTA"])
        # vigilante: un rover detenido varios pasos seguidos replanifica en lugar de quedarse parado
        self.paros_seguidos = self.paros_seguidos + 1 if dec.accion == "detener" else 0
        if self.paros_seguidos >= PASOS_PARADO_MAXIMO:
            dec = Decision("replanificar", 0.0, ["V-PARADO-SIN-AVANCE"])
            self.paros_seguidos = 0
        self.estado = "ALERTA" if dec.accion != "avanzar" or fz.velocidad < 0.5 else "EXPLORACION"

        if any(m.startswith("V-") for m in dec.motivos):
            self.vetos += 1
        if dec.accion == "modo_seguro":
            self.rover.en_modo_seguro = True
            self.estado = "MODO_SEGURO"
        elif dec.accion == "replanificar":
            self._registrar(dec.accion, dec.velocidad, dec.motivos, probs, fz)
            if not self._planear():
                self.estado = "SIN_RUTA"
            return self.estado
        elif dec.accion == "avanzar":
            if paso_fisico(self.mundo, self.rover, rumbo):
                self.ruta.pop(0)
                if self.rover.posicion in self.mundo.sitios:
                    self.visitados.add(self.rover.posicion)
            else:
                self.colisiones += 1
                dec.motivos.append("V-CELDA-BLOQUEADA")
                self._planear()
        self._registrar(dec.accion, dec.velocidad, dec.motivos, probs, fz)
        if self.rover.bateria <= 0 and self.estado not in ESTADOS_TERMINALES:
            self.estado = "FRACASO"
        return self.estado

    def ejecutar(self, max_pasos: int = 400) -> dict:
        while self.estado not in ESTADOS_TERMINALES and self.paso < max_pasos:
            self.paso_mision()
        if self.estado not in ESTADOS_TERMINALES:
            self.estado = "FRACASO"
        return self.resumen()

    def resumen(self) -> dict:
        return {
            "estado": self.estado,
            "pasos": self.paso,
            "sitios_visitados": len(self.visitados),
            "sitios_totales": len(self.mundo.sitios),
            "bateria_final": round(self.rover.bateria, 1),
            "vetos": self.vetos,
            "replanes": self.replanes,
            "colisiones": self.colisiones,
            "barrancos_pisados": int(self.mundo.tipo[self.rover.posicion] == BARRANCO),
        }

    def _celda_observada(self, pos, rumbo, distancia):
        """Celda donde el sensor de proximidad ve el obstáculo, o None si no ve nada."""
        if np.isnan(distancia) or distancia >= 20:
            return None
        df, dc = MOVIMIENTOS[rumbo]
        paso = max(1, round(distancia))
        celda = (pos[0] + df * paso, pos[1] + dc * paso)
        return celda if self.mundo.dentro(celda) else None

    def _actualizar_mapa_estimado(self, pos, rumbo, distancia):
        """El rover marca en su mapa estimado el obstáculo que su sensor de proximidad ve."""
        celda = self._celda_observada(pos, rumbo, distancia)
        if celda is not None:
            self.riesgo[celda] = 1.0

    def _distancia_a_riesgo_en_ruta(self) -> float:
        """Pasos hasta el primer peligro conocido sobre la ruta (20 si la ruta está libre)."""
        for i, celda in enumerate(self.ruta[1:], start=1):
            if self.riesgo[celda] >= 0.5:
                return float(min(i, 20))
        return 20.0

    def _obstaculo_en_ruta(self, pos, rumbo, distancia) -> bool:
        """¿El obstáculo visto cae sobre las próximas celdas de la ruta planeada?"""
        celda = self._celda_observada(pos, rumbo, distancia)
        if celda is None or celda == pos:
            return False
        proximas = self.ruta[1:DISTANCIA_BLOQUEO_RUTA + 1]
        return celda in proximas

    def _quedan_sitios(self) -> bool:
        """¿Quedan sitios en el plan vigente? Un sitio fuera del plan (energía) ya no cuenta."""
        return any(o in self.mundo.sitios and o not in self.visitados for o in self.objetivos)

    def _registrar(self, accion, velocidad, motivos, probs, fz):
        self.log.registrar(
            paso=self.paso,
            estado=self.estado,
            posicion=str(self.rover.posicion),
            bateria=round(self.rover.bateria, 2),
            accion=accion,
            velocidad=round(float(velocidad), 3),
            p_barranco=round(float(probs.get("barranco", 0.0)), 3) if probs else None,
            p_obstaculo=round(float(probs.get("obstáculo", 0.0)), 3) if probs else None,
            reglas=";".join(r[0] for r in fz.reglas_activadas) if fz else "",
            motivos=";".join(motivos),
        )


def forzar_probabilidad(clase: str, confianza: float = 0.95):
    """Intervención de la RNA: fuerza la probabilidad a una clase (libre, obstáculo o barranco)."""
    def fn(probs):
        resto = (1.0 - confianza) / max(len(probs) - 1, 1)
        return {k: (confianza if k == clase else resto) for k in probs}
    return fn


def _rumbo(origen, destino) -> str:
    diferencia = (destino[0] - origen[0], destino[1] - origen[1])
    for rumbo, paso in MOVIMIENTOS.items():
        if paso == diferencia:
            return rumbo
    raise ValueError(f"Las celdas {origen} y {destino} no son adyacentes.")


def camino_valido(ruta, riesgo) -> bool:
    """La ruta debe ser contigua y no pasar por celdas que el mapa estimado marca como peligrosas."""
    if not ruta:
        return False
    for a, b in zip(ruta, ruta[1:]):
        df, dc = b[0] - a[0], b[1] - a[1]
        if max(abs(df), abs(dc)) != 1:              # paso de ocho direcciones
            return False
    return all(riesgo[c] < 0.5 for c in ruta)


def evento_barranco_en_ruta(mision: Mision):
    """Evento: aparece un barranco justo delante de la siguiente celda de la ruta."""
    if len(mision.ruta) > 2:
        celda = mision.ruta[2]
        if celda not in (mision.mundo.inicio, mision.mundo.refugio) and celda not in mision.mundo.sitios:
            mision.mundo.marcar_barranco(celda)
            mision.riesgo[celda] = 1.0

In [7]:
# ----- intervencion.py -----
"""Prueba causal (ARX-10, ev3-d32 A): forzar la salida de la RNA o del difuso y medir qué cambia.

Por cada semilla se corre una misión base (sin intervención) y una misión por variante, con la
misma semilla. Hasta el primer paso con intervención, las dos misiones son idénticas, así que el
primer paso con decisión distinta mide el efecto causal de la intervención.

Variantes:
- rna_barranco: la RNA devuelve probabilidad 0,95 de barranco.
- rna_libre: la RNA devuelve probabilidad 0,95 de libre.
- difuso_min: el difuso devuelve velocidad 0 (parar).
- difuso_max: el difuso devuelve velocidad 1 (máxima).
"""
import pandas as pd


SEMILLAS_POR_DEFECTO = range(20)
MAX_PASOS = 400


def opciones_variantes() -> dict:
    return {
        "rna_barranco": {"intervencion_rna": forzar_probabilidad("barranco")},
        "rna_libre": {"intervencion_rna": forzar_probabilidad("libre")},
        "difuso_min": {"intervencion_difuso": 0.0},
        "difuso_max": {"intervencion_difuso": 1.0},
    }


def _mision(semilla: int, **opciones) -> Mision:
    mundo = generar_mundo(semilla=semilla, n_obstaculos=20, n_barrancos=4, n_sitios=3)
    return Mision(mundo, semilla=semilla, **opciones)


def _primer_cambio(log_base: pd.DataFrame, log_var: pd.DataFrame):
    """Primer paso donde difieren la acción o la velocidad; None si nunca difieren."""
    n = min(len(log_base), len(log_var))
    for i in range(n):
        a, b = log_base.iloc[i], log_var.iloc[i]
        if a["accion"] != b["accion"] or a["velocidad"] != b["velocidad"]:
            return i + 1
    if len(log_base) != len(log_var):
        return n + 1
    return None


def correr_semilla(semilla: int, variante: str, opciones: dict) -> dict:
    """Corre base y variante con la misma semilla y devuelve el efecto medido."""
    base = _mision(semilla)
    res_base = base.ejecutar(max_pasos=MAX_PASOS)
    var = _mision(semilla, **opciones)
    res_var = var.ejecutar(max_pasos=MAX_PASOS)
    cambio = _primer_cambio(base.log.a_dataframe(), var.log.a_dataframe())
    return {
        "variante": variante, "semilla": semilla,
        "estado_base": res_base["estado"], "estado_var": res_var["estado"],
        "exito_base": res_base["estado"] == "EXITO", "exito_var": res_var["estado"] == "EXITO",
        "colisiones": res_var["colisiones"], "barrancos_pisados": res_var["barrancos_pisados"],
        "pasos": res_var["pasos"], "sitios": res_var["sitios_visitados"],
        "paso_primer_cambio": cambio,
    }


def correr(semillas=SEMILLAS_POR_DEFECTO) -> pd.DataFrame:
    """Tabla con una fila por semilla y variante."""
    filas = []
    for nombre, opciones in opciones_variantes().items():
        for s in semillas:
            filas.append(correr_semilla(s, nombre, opciones))
    return pd.DataFrame(filas)


def resumen(tabla: pd.DataFrame) -> pd.DataFrame:
    """Éxito, colisiones, barrancos, pasos, sitios y cambio de decisión por variante."""
    grupos = []
    for nombre, g in tabla.groupby("variante", sort=False):
        cambiaron = g["paso_primer_cambio"].notna()
        grupos.append({
            "variante": nombre,
            "exito_pct": round(100 * g["exito_var"].mean(), 1),
            "colisiones": int(g["colisiones"].sum()),
            "barrancos_pisados": int(g["barrancos_pisados"].sum()),
            "sitios_media": round(g["sitios"].mean(), 2),
            "pasos_mediana": float(g["pasos"].median()),
            "semillas_con_cambio_de_decision": int(cambiaron.sum()),
            "primer_cambio_mediana": (float(g.loc[cambiaron, "paso_primer_cambio"].median())
                                      if cambiaron.any() else None),
        })
    return pd.DataFrame(grupos).set_index("variante")


if __name__ == "__main__":
    tabla = correr()
    print(resumen(tabla).to_string())

              exito_pct  colisiones  barrancos_pisados  sitios_media  pasos_mediana  semillas_con_cambio_de_decision  primer_cambio_mediana
variante                                                                                                                                   
rna_barranco        0.0           0                  0           0.0          400.0                               20                    1.0
rna_libre         100.0           0                  0           1.7           39.5                                0                    NaN
difuso_min          0.0           0                  0           0.0          400.0                               20                    1.0
difuso_max        100.0           0                  0           1.7           39.5                               20                    1.0


## 2. Entrenamiento de la RNA

La RNA se entrena con mapas separados por semilla. Se reporta macro-F1 y recall de barrancos en prueba.

In [8]:
tr, va, te = particion_por_mapas(total_mapas=60)
X_tr, y_tr = generar_dataset(tr, muestras_por_mapa=400)
X_te, y_te = generar_dataset(te, muestras_por_mapa=400)
rna_final = NeuralPerception(semilla=42).entrenar(X_tr, y_tr)
res = rna_final.evaluar(X_te, y_te)
print(f'macro-F1 = {res.macro_f1:.3f}  recall barrancos = {res.recall_barranco:.3f}')
print(f'línea base (clase mayoritaria) = {res.precision_base:.3f}  ECE = {res.error_calibracion:.3f}')
res.matriz_confusion

macro-F1 = 0.928  recall barrancos = 0.943
línea base (clase mayoritaria) = 0.927  ECE = 0.011


array([[3090,    0,    0],
       [   2,  196,   12],
       [   0,    2,   33]])

## 3. Lógica difusa

Ejemplos de inferencia con reglas activadas.

In [9]:
ctrl = FuzzyController()
for caso in [dict(distancia=1, bateria=90, inclinacion=5), dict(distancia=18, bateria=90, inclinacion=2)]:
    r = ctrl.inferir(**caso)
    print(caso, '-> velocidad', round(r.velocidad, 3), [x[0] for x in r.reglas_activadas])

{'distancia': 1, 'bateria': 90, 'inclinacion': 5} -> velocidad 0.053 ['R1']
{'distancia': 18, 'bateria': 90, 'inclinacion': 2} -> velocidad 0.846 ['R4']


## 4. Algoritmo genético

Planificación de una ruta sobre el mapa estimado, con desglose del fitness.

In [10]:
mundo = generar_mundo(semilla=11, n_obstaculos=20, n_barrancos=4, n_sitios=3)
riesgo = mapa_estimado(mundo, np.random.default_rng(11))
res_ag = GeneticPlanner(mundo, riesgo, bateria=100, k_puntos=3, semilla=11).optimizar()
print('fitness', round(res_ag.mejor_desglose.fitness, 3), 'pasos', res_ag.mejor_desglose.ruta_pasos)
res_ag.mejor_desglose

fitness 0.694 pasos 59


Desglose(fitness=0.6938559322033898, ciencia=0.9, distancia=0.09833333333333333, energia=0.59, riesgo=0.04999999999999998, tiempo=0.1864406779661017, penalizacion=0.0, reparaciones=0, ruta_pasos=59)

## 5. Experimento: 20 semillas, híbrido frente a ablaciones

Cada semilla genera un mapa distinto. Se comparan tres variantes con la misma misión:
- **sin RNA** (ablación): la RNA se quita del ciclo.
- **sin difuso** (ablación): la velocidad queda fija en 0,5.

Se reportan éxito, colisiones, barrancos pisados, sitios visitados, pasos y batería, con media, mediana y dispersión.


In [11]:
import pandas as pd

SEMILLAS = range(20)

VARIANTES = {
    "híbrido": {},
    "sin RNA (ablación)": {"usar_rna": False},
    "sin difuso (ablación)": {"usar_difuso": False},
}

filas = []
for nombre, opciones in VARIANTES.items():
    for s in SEMILLAS:
        mundo = generar_mundo(semilla=s, n_obstaculos=20, n_barrancos=4, n_sitios=3)
        r = Mision(mundo, semilla=s, **opciones).ejecutar(max_pasos=400)
        filas.append({"variante": nombre, "semilla": s, **r})

tabla = pd.DataFrame(filas)
tabla["exito"] = tabla["estado"].eq("EXITO").astype(int)
tabla_resumen = tabla.groupby("variante").agg(
    exito_pct=("exito", lambda x: round(100 * x.mean(), 1)),
    colisiones=("colisiones", "sum"),
    barrancos_pisados=("barrancos_pisados", "sum"),
    sitios_media=("sitios_visitados", "mean"),
    pasos_mediana=("pasos", "median"),
    pasos_desv=("pasos", "std"),
    bateria_media=("bateria_final", "mean"),
)
tabla_resumen.round(2)


,exito_pct,colisiones,barrancos_pisados,sitios_media,pasos_mediana,pasos_desv,bateria_media
variante,,,,,,,
híbrido,100.0,0,0,1.7,39.5,7.92,61.1
sin RNA (ablación),100.0,0,0,1.7,39.5,7.92,61.1
sin difuso (ablación),100.0,0,0,1.7,39.5,7.92,61.1


In [12]:
# Detalle por semilla del sistema híbrido
tabla[tabla["variante"] == "híbrido"][["semilla", "estado", "pasos", "sitios_visitados",
                                      "bateria_final", "vetos", "replanes", "colisiones"]]


,semilla,estado,pasos,sitios_visitados,bateria_final,vetos,replanes,colisiones
0,0,EXITO,33,1,68.0,0,1,0
1,1,EXITO,39,1,63.0,1,2,0
2,2,EXITO,30,1,71.0,0,1,0
3,3,EXITO,49,2,52.0,0,1,0
4,4,EXITO,48,2,53.0,0,1,0
5,5,EXITO,38,1,64.0,1,2,0
6,6,EXITO,45,2,57.0,1,2,0
7,7,EXITO,40,1,61.0,0,1,0
8,8,EXITO,42,1,59.0,0,1,0
9,9,EXITO,47,2,55.0,1,2,0


## 5b. Prueba causal: intervención en la RNA y en el difuso (ev3-d32 A)

Se fuerza la salida de la RNA (barranco, libre) y la del difuso (velocidad 0 y velocidad 1). Para cada semilla se compara la misión base con la intervenida, con la misma semilla. El primer paso donde cambia la acción o la velocidad mide el efecto causal.


In [13]:
tabla_intervencion = correr(range(20))
resumen(tabla_intervencion)


,exito_pct,colisiones,barrancos_pisados,sitios_media,pasos_mediana,semillas_con_cambio_de_decision,primer_cambio_mediana
variante,,,,,,,
rna_barranco,0.0,0,0,0.0,400.0,20,1.0
rna_libre,100.0,0,0,1.7,39.5,0,NaN
difuso_min,0.0,0,0,0.0,400.0,20,1.0
difuso_max,100.0,0,0,1.7,39.5,20,1.0


## 6. Una misión paso a paso (bitácora)

Se ejecuta una misión y se muestra su bitácora. Cada fila es un paso del pseudociclo.


In [14]:
mision = Mision(generar_mundo(semilla=11, n_obstaculos=20, n_barrancos=4, n_sitios=3), semilla=11)
mision.ejecutar(max_pasos=400)
bitacora = mision.log.a_dataframe()
print("Resumen:", mision.resumen())
bitacora[["paso", "estado", "posicion", "bateria", "accion", "velocidad", "p_barranco", "motivos"]].head(15)


Resumen: {'estado': 'EXITO', 'pasos': 60, 'sitios_visitados': 3, 'sitios_totales': 3, 'bateria_final': 41.0, 'vetos': 0, 'replanes': 1, 'colisiones': 0, 'barrancos_pisados': 0}


,paso,estado,posicion,bateria,accion,velocidad,p_barranco,motivos
0,1,EXPLORACION,"(19, 1)",99.0,avanzar,0.846,0.000,OK
1,2,EXPLORACION,"(19, 2)",98.0,avanzar,0.846,0.000,OK
2,3,EXPLORACION,"(19, 3)",97.0,avanzar,0.846,0.000,OK
3,4,EXPLORACION,"(19, 4)",96.0,avanzar,0.846,0.000,OK
4,5,EXPLORACION,"(19, 5)",95.0,avanzar,0.846,0.000,OK
5,6,EXPLORACION,"(19, 6)",94.0,avanzar,0.832,0.000,OK
6,7,EXPLORACION,"(19, 7)",93.0,avanzar,0.839,0.002,OK
7,8,ALERTA,"(18, 8)",92.0,avanzar,0.250,0.000,OK
8,9,EXPLORACION,"(18, 9)",91.0,avanzar,0.838,0.000,OK
9,10,EXPLORACION,"(18, 10)",90.0,avanzar,0.846,0.000,OK


In [15]:
# Exportación de la misión: trazas en CSV y configuración con resultados en JSON
import json

bitacora.to_csv("mision_traza.csv", index=False)
with open("mision_resumen.json", "w", encoding="utf-8") as f:
    json.dump({"semilla": 11, "planeta": "Marte", "config": {"n_obstaculos": 20, "n_barrancos": 4,
               "n_sitios": 3, "bateria_inicial": 100.0}, "resultado": mision.resumen(),
               "supuesto": "parámetros de simulación ficticios, no especificaciones NASA"},
              f, ensure_ascii=False, indent=2)
print("Exportado: mision_traza.csv y mision_resumen.json")


Exportado: mision_traza.csv y mision_resumen.json


## 7. Resultados, limitaciones y conclusiones

**Experimento (20 semillas, mapas de Marte, movimiento en 8 direcciones)**
- Híbrido: 100 % de éxito, 0 colisiones, 0 barrancos pisados, 1,7 sitios de media, mediana de 39,5 pasos y batería media de 61,1 %.
- Ablaciones (sin RNA, sin difuso): mismas cifras. Con los sensores activos, la RNA y el difuso no cambian el resultado final en este escenario.
- RNA en prueba: macro-F1 0,928 y recall de barrancos 0,943 (33 de 35). La línea base por clase mayoritaria es 0,927.

**Prueba causal (intervención forzada)**
- RNA forzada a barranco: la decisión cambia en 20 de 20 semillas desde el paso 1; el éxito cae a 0 % porque el supervisor veta sin parar.
- RNA forzada a libre: no cambia ninguna decisión (0 de 20).
- Velocidad del difuso forzada a 0: la decisión cambia en 20 de 20; el éxito cae a 0 %.
- Velocidad del difuso forzada a 1: la decisión cambia en 20 de 20; el éxito sigue en 100 %.

**Combate**
- En modo automático, con dos drones activos, la semilla 11 llega al refugio en 52 pasos, sin colisiones.

**Limitaciones conocidas**
- Falso positivo del sensor: con ruido, una celda libre puede quedar marcada como obstáculo y el plan puede dar SIN_RUTA aunque el mapa real tenga camino.
- El plan omite sitios científicos por energía: se visitan 1,7 de 3 de media.
- El supervisor está congelado y no conoce a los drones: la decisión de combate vive en la capa de la API.
- Los modos rna, difuso y ag muestran sus paneles, pero el paso lo da siempre el pipeline completo.

**Conclusión**
Las tres técnicas están implementadas, probadas por separado y conectadas en el ciclo de decisión con supervisor de seguridad. La evidencia principal es el experimento de 20 semillas y la prueba causal, que muestran que alterar la salida de la RNA o del difuso cambia la decisión, aunque en este escenario el resultado final depende sobre todo del sensor y del supervisor.
